# Relocation Copilot — annotated multi-agent notebook

This notebook demonstrates a stateful relocation planner with **Nebius** for every LLM call, **Mem0** for cross-session user preferences, **LangGraph** for specialist orchestration and human approval, and **LangSmith** for end-to-end evaluations.

The included Bangalore → Seoul evidence is a **synthetic demo packet**, not live legal, housing, daycare, or travel information. Replace it with dated, cited source records before using the output for a real move. No external booking or message is sent.

Run cells in order. Python 3.10+ is required. Set `NEBIUS_API_KEY` and `NEBIUS_MODEL` in your environment. Set `LANGSMITH_API_KEY` to upload evaluations. Model IDs change; choose one currently available in your Nebius account.

## 1. Install dependencies

The notebook keeps state in a local SQLite checkpoint and Mem0's local Qdrant store. Installation needs network access. Restart the kernel after the install cell if imports fail.

In [ ]:
%pip install -q openai mem0ai langgraph langgraph-checkpoint-sqlite langsmith qdrant-client pydantic

## 2. Configure providers

Nebius handles the specialists' calls and Mem0's extraction and embedding calls. Keys are read from environment variables; do not paste secrets into the notebook. The embedding model's output dimension is discovered before creating the Mem0 collection.

In [ ]:
import os, json, time, uuid, sqlite3
from pathlib import Path
from datetime import datetime, timezone
from typing import TypedDict
from openai import OpenAI
from langsmith import Client, traceable
from langsmith.wrappers import wrap_openai
from mem0 import Memory
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt
from langgraph.checkpoint.sqlite import SqliteSaver

ROOT = Path.cwd()
DATA = ROOT / "relocation_data"
DATA.mkdir(exist_ok=True)
NEBIUS_URL = "https://api.tokenfactory.nebius.com/v1/"
NEBIUS_MODEL = os.getenv("NEBIUS_MODEL", "")
EMBED_MODEL = os.getenv("NEBIUS_EMBED_MODEL", "Qwen/Qwen3-Embedding-8B")
NEBIUS_KEY = os.getenv("NEBIUS_API_KEY", "")
assert NEBIUS_KEY and NEBIUS_MODEL, "Set NEBIUS_API_KEY and NEBIUS_MODEL in the environment."
os.environ["LANGSMITH_TRACING"] = "true" if os.getenv("LANGSMITH_API_KEY") else "false"
os.environ["LANGSMITH_PROJECT"] = "relocation-copilot"
client = wrap_openai(OpenAI(base_url=NEBIUS_URL, api_key=NEBIUS_KEY, timeout=45, max_retries=2))
embedding_dims = len(client.embeddings.create(model=EMBED_MODEL, input="dimension probe").data[0].embedding)
print("Nebius model:", NEBIUS_MODEL, "| embedding dimensions:", embedding_dims)

## 3. Initialize Mem0

Mem0 stores durable facts such as the family's budget and commute preference. The LangGraph checkpoint holds the current workflow state. This separation prevents old search results from silently becoming current facts.

In [ ]:
memory = Memory.from_config({
    "llm": {"provider": "openai", "config": {
        "api_key": NEBIUS_KEY, "openai_base_url": NEBIUS_URL,
        "model": NEBIUS_MODEL, "temperature": 0
    }},
    "embedder": {"provider": "openai", "config": {
        "api_key": NEBIUS_KEY, "openai_base_url": NEBIUS_URL,
        "model": EMBED_MODEL, "embedding_dims": embedding_dims
    }},
    "vector_store": {"provider": "qdrant", "config": {
        "collection_name": "relocation_copilot",
        "path": str(DATA / "mem0_qdrant"), "embedding_model_dims": embedding_dims
    }}
})
USER_ID = "demo-family"
memory.add("We are a family of three with a toddler. Our housing ceiling is KRW 4,000,000 per month and preferred daycare commute is at most 30 minutes.", user_id=USER_ID)
print(memory.search("housing budget and daycare commute", user_id=USER_ID))

## 4. Define the input and evidence contract

A source record is a dated observation with an ID and URL. The fixture records use `demo.invalid` deliberately: they are illustrative and cannot support real-world decisions. Replace each record with a working citation and retrieval date. The move year, citizenship, visa status, and work location are deliberately missing, so the plan must request them.

In [ ]:
PROFILE = {
    "origin": "Bangalore", "destination": "Seoul", "move_date": "October 24",
    "family": ["adult", "adult", "toddler"], "citizenship": None,
    "visa_status": None, "workplace": None, "housing_budget_krw": 4000000,
    "max_daycare_commute_min": 30
}
EVIDENCE = [
    {"id":"H1","domain":"housing","title":"Demo apartment A","url":"https://demo.invalid/h1","retrieved":"2026-09-13","facts":{"name":"A","monthly_krw":5000000,"furnished":True,"daycare_commute_min":20}},
    {"id":"H2","domain":"housing","title":"Demo apartment B","url":"https://demo.invalid/h2","retrieved":"2026-09-13","facts":{"name":"B","monthly_krw":3500000,"furnished":False,"daycare_commute_min":45}},
    {"id":"H3","domain":"housing","title":"Demo apartment C","url":"https://demo.invalid/h3","retrieved":"2026-09-13","facts":{"name":"C","monthly_krw":3900000,"furnished":True,"daycare_commute_min":25}},
    {"id":"F1","domain":"family","title":"Demo daycare commute observations","url":"https://demo.invalid/f1","retrieved":"2026-09-13","facts":{"A":20,"B":45,"C":25}},
    {"id":"T1","domain":"travel","title":"Demo travel planning note","url":"https://demo.invalid/t1","retrieved":"2026-09-13","facts":{"booking_requires_visa_clarity":True}}
]
def source_ids():
    return {item["id"] for item in EVIDENCE}
assert all(x["url"].startswith("https://demo.invalid/") for x in EVIDENCE)
print("Synthetic evidence records:", len(EVIDENCE))

## 5. Define shared state and Nebius specialist calls

Every specialist returns structured findings. The prompt limits claims to the supplied evidence and asks for missing information instead of guessing. The retry wrapper allows one retry, then records a recoverable failure.

In [ ]:
class RelocationState(TypedDict, total=False):
    profile: dict
    evidence: list[dict]
    memory_context: list[str]
    findings: dict
    tasks: list[dict]
    conflicts: list[str]
    questions: list[str]
    errors: list[str]
    approval: dict
    report: dict

def call_json(role: str, brief: dict) -> dict:
    prompt = ("You are the " + role + " specialist for a relocation planner. "
              "Return only a JSON object with keys summary (string), actions (array of short strings), "
              "source_ids (array of IDs from the evidence), and questions (array of strings). "
              "Use only the supplied evidence for factual claims. Do not invent requirements, prices, "
              "availability, or citations. If evidence is missing, say so and ask a question.")
    last_error = None
    for attempt in range(2):
        try:
            response = client.chat.completions.create(
                model=NEBIUS_MODEL, temperature=0,
                messages=[{"role":"system","content":prompt},
                          {"role":"user","content":json.dumps(brief, ensure_ascii=False)}]
            )
            raw = response.choices[0].message.content or "{}"
            raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            obj = json.loads(raw)
            if not isinstance(obj, dict) or not isinstance(obj.get("source_ids", []), list):
                raise ValueError("Invalid specialist JSON")
            if not set(obj.get("source_ids", [])).issubset(source_ids()):
                raise ValueError("Specialist cited an unknown source ID")
            return obj
        except Exception as exc:
            last_error = exc
            if attempt == 0: time.sleep(1)
    raise RuntimeError(f"{role} failed after retry: {last_error}")

def specialist(role: str, domain: str):
    def node(state: RelocationState):
        packet = {
            "profile": state["profile"],
            "memory": state.get("memory_context", []),
            "evidence": [e for e in state["evidence"] if e["domain"] == domain],
            "prior_findings": state.get("findings", {})
        }
        try:
            result = call_json(role, packet)
            return {"findings": {**state.get("findings", {}), role: result}}
        except Exception as exc:
            return {"errors": state.get("errors", []) + [str(exc)],
                    "findings": {**state.get("findings", {}), role: {
                        "summary":"Unavailable after retry", "actions":[], "source_ids":[], "questions":[f"Retry {role} research"]
                    }}}
    return node

## 6. Orchestrate agents and resolve cross-agent conflicts

Housing first proposes the cheapest listed candidate. Finance and Family then check budget and daycare commute. If either rejects it, the orchestrator selects a candidate that satisfies both constraints. The housing specialist is called again with the conflict and selected candidate, making the replan visible in the trace.

In [ ]:
def load_memory(state: RelocationState):
    found = memory.search("family housing budget daycare commute", user_id=USER_ID)
    facts = [x.get("memory","") for x in found.get("results", [])]
    return {"memory_context": facts}

def plan(state: RelocationState):
    p = state["profile"]
    questions = []
    for key, wording in [
        ("citizenship","What citizenship(s) and passport validity does each traveler have?"),
        ("visa_status","What is each traveler's visa or residence status?"),
        ("workplace","Where is the workplace or target commute location?")
    ]:
        if not p.get(key): questions.append(wording)
    if len(str(p["move_date"]).split()) < 3:
        questions.append("Which year is the October 24 move?")
    tasks = [
        {"id":"identity","title":"Confirm citizenship, visa status, and move year","depends_on":[],"status":"needs_user"},
        {"id":"immigration","title":"Verify immigration requirements from official sources","depends_on":["identity"],"status":"blocked"},
        {"id":"housing","title":"Compare housing candidates","depends_on":[],"status":"working"},
        {"id":"family","title":"Check daycare commute","depends_on":["housing"],"status":"blocked"},
        {"id":"finance","title":"Check affordability","depends_on":["housing"],"status":"blocked"},
        {"id":"logistics","title":"Adjust shipping to furnished status","depends_on":["housing"],"status":"blocked"},
        {"id":"travel","title":"Plan travel timing","depends_on":["immigration"],"status":"blocked"}
    ]
    return {"questions":questions, "tasks":tasks, "findings":{}, "conflicts":[], "errors":[]}

def reconcile(state: RelocationState):
    p = state["profile"]
    homes = [e["facts"] for e in state["evidence"] if e["domain"] == "housing"]
    proposed = min(homes, key=lambda h: h["monthly_krw"])
    conflicts = []
    if proposed["monthly_krw"] > p["housing_budget_krw"]:
        conflicts.append("Finance rejected housing: above budget")
    if proposed["daycare_commute_min"] > p["max_daycare_commute_min"]:
        conflicts.append("Family rejected housing: daycare commute too long")
    valid = [h for h in homes if h["monthly_krw"] <= p["housing_budget_krw"]
             and h["daycare_commute_min"] <= p["max_daycare_commute_min"]]
    selected = min(valid, key=lambda h:h["monthly_krw"]) if valid else None
    if selected and selected["name"] != proposed["name"]:
        try:
            revised = call_json("housing revision", {
                "profile":p, "initial_candidate":proposed, "conflicts":conflicts,
                "selected_candidate":selected,
                "evidence":[e for e in state["evidence"] if e["domain"]=="housing"]
            })
        except Exception as exc:
            revised = {"summary":"Revision unavailable", "actions":[], "source_ids":[], "questions":[]}
            conflicts.append(str(exc))
    else:
        revised = {}
    tasks = [dict(t) for t in state["tasks"]]
    for t in tasks:
        if t["id"] in ("housing","family","finance","logistics"):
            t["status"] = "done" if selected else "needs_user"
    return {"conflicts":conflicts, "tasks":tasks,
            "findings":{**state["findings"], "housing_revision":revised,
                        "selection":{"proposed":proposed,"selected":selected}}}

def verify(state: RelocationState):
    issues = list(state.get("errors", []))
    for name, finding in state["findings"].items():
        if isinstance(finding, dict) and not set(finding.get("source_ids",[])).issubset(source_ids()):
            issues.append(f"{name}: unsupported citation")
    selected = state["findings"]["selection"]["selected"]
    if selected is None: issues.append("No housing option satisfies both constraints.")
    if selected and selected["monthly_krw"] > state["profile"]["housing_budget_krw"]:
        issues.append("Selected home exceeds budget.")
    report = {
        "profile":state["profile"], "tasks":state["tasks"], "selected_home":selected,
        "conflicts":state["conflicts"], "questions":state["questions"],
        "issues":issues, "findings":state["findings"],
        "evidence_label":"SYNTHETIC DEMO — not verified real-world information"
    }
    return {"report":report}

def approval_node(state: RelocationState):
    selected = state["report"]["selected_home"]
    decision = interrupt({
        "question":"Approve this housing candidate for further research?",
        "candidate":selected, "note":"No booking, payment, or message will be sent."
    })
    return {"approval":{"approved": bool(decision), "candidate":selected["name"] if selected else None}}

## 7. Compile the stateful graph

The SQLite checkpointer allows the approval step to resume after an interrupted run. A distinct `thread_id` isolates each evaluation scenario.

In [ ]:
builder = StateGraph(RelocationState)
builder.add_node("memory", load_memory)
builder.add_node("plan", plan)
for name, domain in [
    ("immigration","immigration"), ("housing","housing"), ("family","family"),
    ("finance","finance"), ("logistics","logistics"), ("travel","travel")
]:
    builder.add_node(name, specialist(name, domain))
builder.add_node("reconcile", reconcile)
builder.add_node("verify", verify)
builder.add_node("approval", approval_node)
order = ["memory","plan","immigration","housing","family","finance","logistics","travel","reconcile","verify","approval"]
builder.add_edge(START, order[0])
for first, second in zip(order, order[1:]): builder.add_edge(first, second)
builder.add_edge("approval", END)
checkpoint_conn = sqlite3.connect(DATA / "checkpoints.sqlite", check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(checkpoint_conn))
print(graph.get_graph().draw_mermaid())

## 8. Run the demo and inspect the approval queue

The graph stops before recording the user's decision. Review the report and then resume with `True` or `False`. Approval records a choice in this notebook; it does not perform any external write.

In [ ]:
demo_config = {"configurable":{"thread_id":"demo-" + uuid.uuid4().hex}}
demo = graph.invoke({"profile":PROFILE, "evidence":EVIDENCE}, config=demo_config)
print(json.dumps(demo["report"], indent=2, ensure_ascii=False))
print("Pending approval:", demo.get("__interrupt__"))
assert demo["report"]["selected_home"]["name"] == "C"
assert demo["report"]["conflicts"]

In [ ]:
# Change True to False to reject this candidate.
final = graph.invoke(Command(resume=True), config=demo_config)
print("Decision:", final["approval"])
print("Task states:", [(t["id"], t["status"]) for t in final["tasks"]])

## 9. Exercise failure recovery

This probe replaces one model call with a deliberate error. The specialist retries once, records the failure, and the rest of the graph continues. It does not need an API key beyond the setup cell and does not alter the real Nebius client.

In [ ]:
original_call = call_json
def forced_failure(role, brief):
    if role == "travel": raise RuntimeError("Simulated travel API outage")
    return original_call(role, brief)
call_json = forced_failure
try:
    failure_config = {"configurable":{"thread_id":"failure-" + uuid.uuid4().hex}}
    failure = graph.invoke({"profile":PROFILE,"evidence":EVIDENCE}, config=failure_config)
    assert any("Simulated travel API outage" in e for e in failure["errors"])
    assert failure["report"]["selected_home"]["name"] == "C"
    print("Recovered with recorded error:", failure["errors"])
finally:
    call_json = original_call

## 10. Evaluate the complete workflow in LangSmith

The dataset contains five scenarios, including an over-budget proposal, a daycare conflict, missing identity information, an impossible housing constraint, and a tool outage. Evaluators check observable behavior rather than grading the model's prose. Uploading requires `LANGSMITH_API_KEY`. Each target run gets a new thread ID so checkpoint state cannot leak between examples.

In [ ]:
EVAL_CASES = [
    {"name":"base","profile":PROFILE,"expected_home":"C","expect_question":True,"force_travel_failure":False},
    {"name":"higher_budget","profile":{**PROFILE,"housing_budget_krw":5500000},"expected_home":"B","expect_question":True,"force_travel_failure":False},
    {"name":"strict_daycare","profile":{**PROFILE,"max_daycare_commute_min":22,"housing_budget_krw":5500000},"expected_home":"A","expect_question":True,"force_travel_failure":False},
    {"name":"no_valid_home","profile":{**PROFILE,"housing_budget_krw":3000000},"expected_home":None,"expect_question":True,"force_travel_failure":False},
    {"name":"travel_failure","profile":PROFILE,"expected_home":"C","expect_question":True,"force_travel_failure":True}
]
def eval_target(inputs: dict) -> dict:
    global call_json
    saved = call_json
    if inputs["force_travel_failure"]:
        def outage(role, brief):
            if role == "travel": raise RuntimeError("Simulated travel API outage")
            return saved(role, brief)
        call_json = outage
    try:
        config = {"configurable":{"thread_id":"eval-" + uuid.uuid4().hex}}
        result = graph.invoke({"profile":inputs["profile"],"evidence":EVIDENCE},config=config)
        report = result["report"]
        return {
            "selected_home":(report["selected_home"] or {}).get("name"),
            "has_questions":bool(report["questions"]),
            "conflicts":report["conflicts"], "errors":report["issues"],
            "paused_for_approval":bool(result.get("__interrupt__")),
            "evidence_label":report["evidence_label"]
        }
    finally:
        call_json = saved

def selection_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    return {"key":"correct_selection","score":int(outputs["selected_home"] == reference_outputs["expected_home"])}
def approval_evaluator(outputs: dict) -> dict:
    return {"key":"approval_pause","score":int(outputs["paused_for_approval"])}
def grounding_evaluator(outputs: dict) -> dict:
    return {"key":"demo_evidence_labeled","score":int("SYNTHETIC DEMO" in outputs["evidence_label"])}
def failure_evaluator(outputs: dict, reference_outputs: dict) -> dict:
    expected = reference_outputs["force_travel_failure"]
    observed = any("Simulated travel API outage" in e for e in outputs["errors"])
    return {"key":"failure_recovered","score":int(observed == expected)}

In [ ]:
if os.getenv("LANGSMITH_API_KEY"):
    ls = Client()
    dataset_name = "relocation-copilot-mvp-v1"
    try:
        dataset = ls.read_dataset(dataset_name=dataset_name)
    except Exception:
        dataset = ls.create_dataset(dataset_name=dataset_name,
                                    description="Synthetic relocation coordination scenarios")
        ls.create_examples(
            dataset_id=dataset.id,
            inputs=[{k:v for k,v in c.items() if k not in ("name","expected_home","expect_question")} for c in EVAL_CASES],
            outputs=[{"expected_home":c["expected_home"],"force_travel_failure":c["force_travel_failure"]} for c in EVAL_CASES]
        )
    results = ls.evaluate(
        eval_target, data=dataset_name,
        evaluators=[selection_evaluator,approval_evaluator,grounding_evaluator,failure_evaluator],
        experiment_prefix="relocation-notebook", max_concurrency=1
    )
    print("LangSmith evaluation complete:", results)
else:
    print("Set LANGSMITH_API_KEY to create the dataset and upload an evaluation experiment.")

## 11. Next build iteration

This notebook demonstrates orchestration and state, but its evidence packet is synthetic. A production-oriented version needs a source-retrieval tool, official immigration citations, dated housing and daycare records, stronger privacy controls, and a user interface. The notebook currently never books or sends anything.

**Integration references:** [Nebius API](https://docs.tokenfactory.nebius.com/api-reference/introduction), [Nebius embeddings](https://docs.tokenfactory.nebius.com/api-reference/examples/create-embeddings), [Mem0 configuration](https://docs.mem0.ai/open-source/configuration), [Mem0 LLM configuration](https://docs.mem0.ai/components/llms/config), [LangGraph interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts), [LangSmith evaluation](https://docs.langchain.com/langsmith/evaluation-quickstart).